# Notebook 05 — Discusión y conclusiones

---

### Entorno y persistencia de resultados

La celda siguiente fija tres cosas que condicionan la reproducibilidad del experimento:

**Semilla fija.** `RANDOM_STATE = 42` se aplica a la partición train/test y al ajuste de todos los
modelos. Sin ella, cada ejecución produciría particiones distintas y las métricas no serían
comparables entre corridas ni verificables por un tercero.

**Persistencia en Google Drive.** Los resultados se escriben en `MyDrive/hotel_booking` y no en el
disco temporal de Colab, que se borra al cerrar la sesión. Esto permite que el experimento se ejecute
en varias sesiones sin repetir etapas: el notebook 03 deja las particiones preparadas y los notebooks
04 y 05 las consumen tal cual, garantizando que todos operan exactamente sobre los mismos datos.
Si el montaje no se completa, la celda interrumpe la ejecución en lugar de escribir en una ubicación
volátil.

**Registro de las figuras.** La función `guardar` escribe cada gráfico en `splits/` como PNG a 200
dpi. Se invoca siempre antes de `plt.show()`, porque mostrar la figura vacía el buffer de matplotlib
y el archivo resultante quedaría en blanco.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = "/content/drive/MyDrive/hotel_booking"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath("./hotel_booking")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)

def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=200, bbox_inches="tight", facecolor="white")
    print("Grafico guardado:", destino)

print("Guardando en Google Drive" if EN_DRIVE else "Google Drive no disponible: guardando local")
print("Carpeta de trabajo:", RUTA)
print("Graficos (splits) :", CARPETA_SPLITS)

In [ ]:
import joblib
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

paquete = joblib.load(os.path.join(RUTA, "datos_preparados.joblib"))
salida = joblib.load(os.path.join(RUTA, "modelo_y_resultados.joblib"))

X_train, X_test = paquete["X_train"], paquete["X_test"]
y_train, y_test = paquete["y_train"], paquete["y_test"]
modelo_final = salida["modelo_final"]
resultados = salida["resultados"]
tabla_peso = salida["tabla_peso"]
pred_train, pred_test = salida["pred_train"], salida["pred_test"]
etiquetas = salida["etiquetas"]

print("Mejor configuracion:", salida["mejor"]["estrategia"], "| C =", salida["mejor"]["C"])

## 4.1 Interpretación y comparación de métricas train vs. test (6 %)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, estrategia in zip(axes, ["Softmax", "One-vs-Rest"]):
    sub = resultados[resultados["estrategia"] == estrategia].sort_values("C")
    ax.plot(sub["C"], sub["f1_macro_train"], "o-", label="Train")
    ax.plot(sub["C"], sub["f1_macro_test"], "s-", label="Test")
    ax.set_xscale("log")
    ax.set_xlabel("C  (menor = mas regularizacion)")
    ax.set_ylabel("F1 macro")
    ax.set_title(f"{estrategia}: regularizacion vs desempeno")
    ax.legend()

plt.tight_layout()
guardar("05_regularizacion_vs_desempeno")
plt.show()

print(resultados.sort_values(["estrategia", "C"])[
    ["estrategia", "C", "f1_macro_train", "f1_macro_test", "brecha"]
].round(4).to_string(index=False))

In [ ]:
print("=== Accuracy frente a F1 macro en el modelo elegido ===")
print(f"Accuracy test : {accuracy_score(y_test, pred_test):.4f}")
print(f"F1 macro test : {f1_score(y_test, pred_test, average='macro'):.4f}")
print("\n=== Efecto de la ponderacion por clase ===")
print(tabla_peso.round(4).to_string(index=False))

**Qué interpretar (redactá con tus números).**

- **Cada métrica en contexto.** El *recall* de `No-Show` indica qué proporción de las reservas que
  efectivamente terminaron en no-show el modelo logró anticipar: es la métrica que el hotel usaría
  para decidir a quién exigir garantía. La *precision* de esa misma clase indica cuántas de las
  reservas marcadas como riesgosas lo eran de verdad, y se traduce en cuántos clientes confiables
  serían molestados innecesariamente. Hay un intercambio directo entre ambas.
- **Accuracy contra F1 macro.** La accuracy es notablemente más alta porque está dominada por las dos
  clases abundantes. Esa brecha entre ambas métricas es la evidencia numérica de por qué se eligió
  F1 macro desde la formulación del problema.
- **Diagnóstico de sobreajuste.** Mirá la columna `brecha`. Con `C` chico el modelo está muy
  regularizado y ambas curvas quedan bajas y juntas: **underfitting**. Al aumentar `C` la curva de
  train sube y, si se despega de la de test, aparece **overfitting**. En un modelo lineal con este
  número de variables lo habitual es que la brecha se mantenga pequeña: la regresión logística tiene
  capacidad limitada, y el riesgo real acá es quedarse corto, no pasarse.
- **Cómo mitigarlo.** Si la brecha creciera, la vía es reducir `C` o pasar a penalización L1 con el
  solver `saga`, que además elimina variables poniendo sus coeficientes en cero.

## 4.2 Análisis de la matriz de confusión y errores por clase (6 %)

In [ ]:
cm_norm = confusion_matrix(y_test, pred_test, labels=etiquetas, normalize="true")

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="RdYlGn",
            xticklabels=etiquetas, yticklabels=etiquetas, ax=ax, vmin=0, vmax=1)
ax.set_title("Matriz de confusion normalizada por fila (TEST)")
ax.set_xlabel("Prediccion"); ax.set_ylabel("Valor real")
plt.tight_layout()
guardar("05_matriz_confusion_normalizada")
plt.show()

print("Cada fila suma 100 %: muestra en que se convirtio cada clase real.\n")
print(pd.DataFrame(
    confusion_matrix(y_test, pred_test, labels=etiquetas),
    index=[f"real {e}" for e in etiquetas],
    columns=[f"pred {e}" for e in etiquetas],
))

### El costo de las alertas de No-Show

La diagonal cuenta los aciertos, pero el hotel convive con la **columna** completa: todas las
reservas a las que el modelo le pondría una condición extra.

In [ ]:
cm = confusion_matrix(y_test, pred_test, labels=etiquetas)
i_ns = etiquetas.index("No-Show")
columna = cm[:, i_ns]
total_alertas = columna.sum()

print(f"Reservas marcadas como riesgo de No-Show: {total_alertas}")
for j, e in enumerate(etiquetas):
    print(f"   de las cuales eran realmente {e:<10}: {columna[j]:>5}  ({columna[j]/total_alertas:.1%})")

print(f"\nNo-Shows reales en el test: {cm[i_ns].sum()}")
print(f"Detectados: {cm[i_ns, i_ns]} ({cm[i_ns, i_ns]/cm[i_ns].sum():.1%})")
print(f"\nPor cada no-show detectado, el hotel molesta a "
      f"{(total_alertas - cm[i_ns, i_ns]) / cm[i_ns, i_ns]:.0f} clientes que si habrian venido "
      f"o que habrian avisado.")

**Cómo usar este número.** Es el argumento más concreto que podés llevar a la exposición: traduce
una métrica abstracta (precisión de 0,04) a una consecuencia operativa. Que sea aceptable o no
depende de cuánto cuesta una noche perdida frente a cuánto cuesta pedirle garantía de tarjeta a un
cliente que iba a venir. El modelo no responde eso; lo responde el hotel.

**Qué analizar (redactá con tus números).**

- **Qué clases se confunden, y en qué dirección.** Leé la fila de `No-Show` real. El error dominante
  **no** va hacia `Canceled` como podría suponerse, sino hacia `Check-Out`. Es coherente con la
  estadística descriptiva del notebook 02: una reserva que terminará en no-show tiene la anticipación
  más corta de las tres clases y un historial sin cancelaciones, o sea que **al momento de reservar
  se parece a una reserva normal**, no a una que se va a cancelar.
- **Por qué eso tiene sentido de dominio.** Cancelar es una decisión que el huésped toma y comunica,
  y deja rastro en variables observables (mucha anticipación, historial de cancelaciones, canal de
  agencia). No presentarse es una decisión tomada **después** de reservar, sin aviso, y por lo tanto
  sin huella en los datos de la reserva. El modelo no puede anticipar lo que todavía no ocurrió.
- **La única señal que el modelo encuentra.** Revisá el coeficiente de `country_PRT` para `No-Show`
  en la sección siguiente. Portugal es el mercado local de estos hoteles, y un huésped que vive cerca
  pierde mucho menos por no aparecer que uno que ya pagó un vuelo.
- **El costo de las falsas alarmas.** Mirá la columna `pred No-Show` completa, no solo su diagonal.
  El modelo marca miles de reservas como riesgo de no-show y acierta en una fracción muy pequeña.
  Cuantificá eso: cuántas de las reservas señaladas eran realmente no-shows. Ese número es lo que
  el hotel tendría que aceptar como precio de detectar la clase.
- **Vínculo con el desbalance.** Con ~1 % de representación, `No-Show` aporta muy pocos ejemplos para
  estimar su frontera. `class_weight="balanced"` compensa el peso pero no crea información nueva.
- **Limitación del modelo lineal.** La regresión logística traza fronteras que son hiperplanos. Si la
  distinción dependiera de interacciones —por ejemplo, mucha anticipación **y además** sin depósito
  **y además** cliente no recurrente— un modelo lineal solo puede capturarlas si esas interacciones
  se construyen explícitamente como variables nuevas.

### Coeficientes: qué aprendió el modelo

La ventaja del modelo lineal es que cada variable tiene un peso legible. Un coeficiente positivo para
una clase significa que esa variable empuja la predicción hacia ella.

In [ ]:
try:
    nombres = modelo_final.named_steps["prep"].get_feature_names_out()
    clf = modelo_final.named_steps["clf"]
    coefs = clf.coef_ if hasattr(clf, "coef_") else np.vstack([e.coef_ for e in clf.estimators_])
    clases = clf.classes_ if hasattr(clf, "classes_") else modelo_final.classes_

    tabla_coef = pd.DataFrame(coefs.T, index=nombres, columns=clases)
    for clase in clases:
        print(f"\n=== Variables que mas empujan hacia {clase} ===")
        print(tabla_coef[clase].sort_values(ascending=False).head(8).round(3))

    print("\n=== Donde queda lead_time ===")
    print(tabla_coef.loc[[i for i in tabla_coef.index if "lead_time" in i]].round(3))
except Exception as e:
    print("No se pudieron extraer los coeficientes:", e)

**Tres lecturas que conviene desarrollar en el informe:**

1. **`required_car_parking_spaces` es el predictor más fuerte de `Check-Out`.** Quien reserva
   estacionamiento ya planificó cómo llegar. Es una señal de compromiso que el hotel tiene desde el
   momento de la reserva.
2. **`deposit_type = Non Refund` empuja fuertemente hacia `Canceled`, no hacia `Check-Out`.**
   Es contraintuitivo y vale la pena discutirlo: no significa que exigir depósito cause cancelaciones,
   sino que el hotel probablemente ya exige depósito no reembolsable a las reservas que considera
   riesgosas. El modelo está capturando la política del hotel, no una relación causal. Es un buen
   ejemplo de por qué un coeficiente legible permite detectar este tipo de confusión, cosa que un
   modelo opaco no permitiría.
3. **`lead_time` tiene coeficiente positivo hacia `Canceled` y negativo hacia `Check-Out`**, en la
   dirección que anticipaba el análisis descriptivo, aunque con magnitud menor que las dos anteriores.
4. **`country = PRT` es el predictor más fuerte de `No-Show`.** Portugal es el mercado local de estos
   dos hoteles, y tiene sentido: un huésped que vive cerca pierde menos por no presentarse, mientras
   que quien viajó desde otro país ya asumió el costo del vuelo. Es el único indicio que el modelo
   encuentra para esa clase, y explica por qué su capacidad de detectarla es tan limitada: la
   nacionalidad sola no alcanza.

## 5.1 Conclusiones (8 %)

Redactá las conclusiones con tus resultados. Estas son las cuatro líneas que las sostienen, cada una
ligada a un objetivo específico del notebook 01:

1. **Sobre el objetivo general.** El modelo lineal distingue de forma útil entre reservas que se
   concretan y reservas que se cancelan, empleando solo información disponible al momento de reservar.
   Indicá el F1 macro alcanzado y contrastalo con el 33 % que obtendría una asignación al azar entre
   tres clases.

2. **Sobre las variables más informativas.** Comentá los tres hallazgos del anexo de coeficientes,
   todos accionables porque el hotel los conoce al recibir la reserva: el estacionamiento como señal
   de compromiso, el depósito no reembolsable como marca de la propia política del hotel, y
   `lead_time` en la dirección esperada.

3. **Sobre la clase minoritaria.** `No-Show` no se predice de forma confiable ni siquiera con
   ponderación por clase. Su causa no es solo el 1 % de representación: el análisis descriptivo
   mostró que una reserva destinada a no-show **es indistinguible de una reserva normal** en el
   momento de tomarla, porque la decisión de no presentarse se toma después y no deja rastro en los
   datos disponibles. Es una limitación del problema, no del modelo lineal.

4. **Sobre la utilidad práctica.** Aun sin resolver `No-Show`, el modelo sirve para decidir a qué
   reservas exigir depósito, porque identifica el perfil de cancelación con anticipación suficiente.

**Limitaciones del enfoque lineal.** La regresión logística asume fronteras que son hiperplanos y no
captura interacciones entre variables salvo que se construyan a mano. Comentá también dos cosas que
se ven en el barrido: **One-vs-Rest supera consistentemente a Softmax**, aunque por un margen
modesto, y **el valor de `C` casi no altera el resultado**. Esto último indica que el modelo está
saturado por la representación de los datos y no por la regularización: agregar flexibilidad no
ayuda porque la información que falta no está en las variables disponibles.

**Recomendaciones de mejora.**

- Incorporar variables de comportamiento previo a la llegada (confirmaciones, contactos, cambios
  tardíos), que es la información que probablemente separa `No-Show` de `Canceled`.
- Construir términos de interacción, por ejemplo `lead_time × deposit_type`, para dar al modelo
  lineal acceso a relaciones combinadas sin abandonar su interpretabilidad.
- Ajustar el umbral de decisión de `No-Show` según el costo real de cada tipo de error, en lugar de
  usar el criterio de máxima probabilidad.

---

## Gráficos generados

Los siete gráficos de la serie quedan guardados en PNG a 200 dpi dentro de
`MyDrive/hotel_booking/splits/`, listos para insertar en el informe escrito.
La celda siguiente verifica que estén todos.

In [ ]:
esperados = [
    "02_boxplots_por_clase",
    "02_leadtime_y_tipo_deposito",
    "03_distribucion_de_clases",
    "04_matriz_confusion_train",
    "04_matriz_confusion_test",
    "05_regularizacion_vs_desempeno",
    "05_matriz_confusion_normalizada",
]

print("Carpeta:", CARPETA_SPLITS, "\n")
faltan = []
for nombre in esperados:
    ruta = os.path.join(CARPETA_SPLITS, nombre + ".png")
    if os.path.exists(ruta):
        print(f"  OK     {nombre}.png  ({os.path.getsize(ruta)/1024:.0f} KB)")
    else:
        print(f"  FALTA  {nombre}.png  -> ejecuta el notebook {nombre[:2]}")
        faltan.append(nombre)

print()
print(f"{len(esperados) - len(faltan)} de {len(esperados)} graficos disponibles.")